# Preprocessing Simple — Cleaning Data TravelFit (443 Destinasi)

Fokus hanya pada inti cleaning. Sumber: 437 Kaggle + 6 tambahan Jawa. Aturan utama: **tidak ada destinasi yang dihapus** (wajib 443 ID unik 1-443).
Output: `data/processed/destinations_clean_java443.csv` + `destinations_kmeans_features_java443.csv`.

## Langkah
1. Setup + load 4 sumber
2. Cleaning kolom, teks, angka (sekaligus flag kualitas)
3. Missing value (median hanya untuk Tahura) + dedup interaksi
4. Standardisasi wilayah/kategori + validasi geo + cap harga P99
5. Fitur fasilitas C4 + activity tags
6. Fitur K-Means + simpan CSV


**Budget aplikasi:** batas harga tiket masuk satu destinasi per orang. C1 TOPSIS menggunakan harga asli; cap p99 hanya untuk fitur K-Means. C3 dihitung saat rekomendasi dengan Haversine (garis lurus, bukan jarak jalan). Transportasi, makan dan penginapan tidak dihitung. Semua 443 ID tetap dipertahankan.


In [ ]:
from pathlib import Path
import re, json
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
print('PROJECT_ROOT:', PROJECT_ROOT)

SRC_BASE = 'data/raw/tourism_with_id.csv'
SRC_TAMBAHAN = 'data/review/gabungan_jawa/destinasi_jawa_review.csv'
SRC_RECEIPT = 'data/review/gabungan_jawa/google_maps_rating_review.json'
SRC_INTERAKSI = 'data/raw/tourism_rating.csv'
OUT_DEST = 'data/processed/destinations_clean_java443.csv'
OUT_FEAT = 'data/processed/destinations_kmeans_features_java443.csv'

def snake_case(v):
    return re.sub(r'[^0-9a-zA-Z]+', '_', str(v).strip()).strip('_').lower()

def normalize_text(v):
    return re.sub(r'\s+', ' ', str(v).casefold()).strip() if pd.notna(v) else ''

def contains_keyword(text, keyword):
    esc = re.escape(keyword.casefold()).replace(r'\ ', r'\s+')
    return re.search(r'(?<!\w)' + esc + r'(?!\w)', text) is not None

def check_ids(frame, label='data'):
    assert len(frame) == 443 and frame['place_id'].is_unique and set(frame['place_id']) == set(range(1, 444)), label + ': wajib 443 ID unik 1-443'

print('pandas', pd.__version__, '| numpy', np.__version__)


## 1. Load 4 sumber
Gagal cepat bila ada file hilang atau jumlah ID tidak sesuai (437 ID 1-437, tambahan 6 ID 438-443).


In [ ]:
for rel in [SRC_BASE, SRC_TAMBAHAN, SRC_RECEIPT, SRC_INTERAKSI]:
    assert (PROJECT_ROOT / rel).is_file(), 'hilang: ' + rel
base = pd.read_csv(PROJECT_ROOT / SRC_BASE)
tambahan = pd.read_csv(PROJECT_ROOT / SRC_TAMBAHAN)
interaksi_raw = pd.read_csv(PROJECT_ROOT / SRC_INTERAKSI)
receipt = json.loads((PROJECT_ROOT / SRC_RECEIPT).read_text(encoding='utf-8'))
assert len(base) == 437 and set(base['Place_Id']) == set(range(1, 438))
assert len(tambahan) == 6 and set(tambahan['Place_Id']) == set(range(438, 444))
print('base:', base.shape, '| tambahan:', tambahan.shape, '| interaksi:', interaksi_raw.shape)


## 2. Cleaning kolom + teks + angka
Tiga pekerjaan dalam satu alur: (a) tambah provenance lalu gabung 437+6, terapkan rating Google Maps yang lolos verifikasi, snake_case header, buang `Unnamed*`; (b) rapikan whitespace teks; (c) coerce numerik + flag `data_quality_issues` per baris + standardisasi `province` / `category_clean`.


In [ ]:
# --- 2a. Gabung sumber ---
base['source_dataset'] = 'Indonesia Tourism Destination'
base['source_url'] = 'https://raw.githubusercontent.com/akhiyarwaladi/indonesia_tourism/main/tourism_with_id.csv'
base['rating_status'] = 'dataset_original'
base['rating_source_url'] = base['source_url']
base['rating_access_date'] = ''
tambahan['source_dataset'] = 'Tambahan Jawa: tarif peraturan daerah dan lokasi OSM'
tambahan['source_url'] = 'data/review/gabungan_jawa/Dokumentasi.md'
tambahan['rating_status'] = 'missing'
tambahan['rating_source_url'] = ''
tambahan['rating_access_date'] = ''
for rec in receipt['records']:
    m = tambahan['Place_Id'].eq(rec['place_id']) & tambahan['Place_Name'].eq(rec['dataset_name'])
    if rec.get('integration_allowed') and rec.get('match_status') == 'matched_name_location':
        tambahan.loc[m, 'Rating'] = rec['rating_observed']
        tambahan.loc[m, 'rating_status'] = 'google_maps_matched'
        tambahan.loc[m, 'rating_source_url'] = rec['maps_url']
        tambahan.loc[m, 'rating_access_date'] = receipt.get('access_date', '')
    else:
        tambahan.loc[m, 'rating_status'] = 'pending_identity'
df = pd.concat([base, tambahan], ignore_index=True)
df.columns = [snake_case(c) for c in df.columns]
df = df.drop(columns=[c for c in df.columns if c.startswith('unnamed')])
df = df.sort_values('place_id').reset_index(drop=True)
check_ids(df, 'gabungan')

# --- 2b. Teks: collapse whitespace ---
for col in ['place_name', 'description', 'category', 'city']:
    df[col] = df[col].astype('string').str.replace(r'\s+', ' ', regex=True).str.strip()

# --- 2c. Angka + flag kualitas ---
for col in ['place_id', 'price', 'rating', 'time_minutes', 'lat', 'long']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df['coordinate_review_required'] = (df['place_id'].eq(9) & df['place_name'].eq('Pelabuhan Marina')
    & np.isclose(df['lat'], 1.07888) & np.isclose(df['long'], 103.931398))
df['coordinate_review_note'] = ''
df.loc[df['coordinate_review_required'], 'coordinate_review_note'] = (
    'Label Jakarta bertentangan dengan koordinat sumber di luar Jawa; baris dipertahankan sesuai keputusan pengguna.')
CHECKS = {'id_missing': df['place_id'].isna(), 'id_duplicate': df['place_id'].duplicated(keep=False),
    'name_missing': df['place_name'].isna() | df['place_name'].eq(''), 'price_invalid': ~df['price'].ge(0),
    'rating_missing': df['rating'].isna(),
    'rating_out_of_range': df['rating'].notna() & ~df['rating'].between(1, 5),
    'latitude_invalid': ~df['lat'].between(-90, 90), 'longitude_invalid': ~df['long'].between(-180, 180),
    'coordinate_location_conflict': df['coordinate_review_required']}
df['data_quality_issues'] = ''
for issue, mask in CHECKS.items():
    df.loc[mask.fillna(False), 'data_quality_issues'] += issue + '|'
df['data_quality_issues'] = df['data_quality_issues'].str.rstrip('|')

# --- 2d. Wilayah & kategori ---
CITY_TO_PROVINCE = {'Jakarta': 'DKI Jakarta', 'Bandung': 'Jawa Barat', 'Semarang': 'Jawa Tengah',
    'Yogyakarta': 'DI Yogyakarta', 'Surabaya': 'Jawa Timur', 'Pandeglang': 'Banten',
    'Lebak': 'Banten', 'Purworejo': 'Jawa Tengah', 'Ngawi': 'Jawa Timur'}
CATEGORY_MAP = {'budaya': 'budaya', 'taman hiburan': 'hiburan', 'cagar alam': 'alam',
    'pusat perbelanjaan': 'belanja', 'tempat ibadah': 'religi', 'bahari': 'bahari'}
df['province'] = df['city'].map(CITY_TO_PROVINCE).astype('string')
df['category_original'] = df['category']
df['category_clean'] = df['category'].str.casefold().map(CATEGORY_MAP).astype('string')
assert df['province'].notna().all() and df['category_clean'].notna().all()
print('bersih:', df.shape, '| ber-flag kualitas:', int((df['data_quality_issues'] != '').sum()))
display(df[['place_id', 'place_name', 'province', 'category_clean', 'price', 'rating', 'data_quality_issues']].tail(7))


## 3. Missing value + interaksi pengguna
Rating sumber Tahura (ID438) dipertahankan NaN; kolom model `c2_rating_for_model` diisi median 442 teramati. Interaksi: buang baris invalid, dedup user x place (keep-last), agregat mean/count/std, left-merge.


In [ ]:
median_rating = float(df.loc[df['rating'].between(1, 5), 'rating'].median())
df['c2_rating'] = df['rating'].astype(float)
df['c2_rating_for_model'] = df['c2_rating'].where(df['c2_rating'].between(1, 5))
df['rating_imputed'] = (df['place_id'].eq(438) & df['place_name'].eq('Taman Hutan Raya Banten') & df['rating'].isna())
df.loc[df['rating_imputed'], 'c2_rating_for_model'] = median_rating
df['rating_model_note'] = ''
df.loc[df['rating_imputed'], 'rating_model_note'] = 'Imputasi median rating teramati; bukan rating destinasi yang terverifikasi.'
print('median:', median_rating, '| diimputasi:', df.loc[df['rating_imputed'], 'place_id'].tolist())

rt = interaksi_raw.copy()
rt.columns = [snake_case(c) for c in rt.columns]
for c in ['user_id', 'place_id', 'place_ratings']:
    rt[c] = pd.to_numeric(rt[c], errors='coerce')
rt = rt.loc[rt['user_id'].notna() & rt['place_id'].notna() & rt['place_ratings'].between(1, 5)]
rt = rt.drop_duplicates(['user_id', 'place_id'], keep='last')
agg = rt.groupby('place_id', as_index=False).agg(user_rating_mean=('place_ratings', 'mean'),
    user_rating_count=('place_ratings', 'count'), user_rating_std=('place_ratings', 'std'))
agg['user_rating_std'] = agg['user_rating_std'].fillna(0)
df = df.merge(agg, on='place_id', how='left', validate='one_to_one')
df['c1_ticket_price'] = df['price'].astype(float)
check_ids(df, 'post-merge')
print('interaksi valid:', len(rt), '| destinasi tercakup:', agg['place_id'].nunique())


## 4. Geo + harga


In [ ]:
assert df['lat'].between(-90, 90).all() and df['long'].between(-180, 180).all()
assert (df['price'] >= 0).all() and df['price'].notna().all()
cap_p99 = float(np.percentile(df['price'].to_numpy(dtype=float), 99))
print('P99: Rp', format(cap_p99, ',.0f'), '| max master: Rp', format(df['price'].max(), ',.0f'))
print('koordinat review:', df.loc[df['coordinate_review_required'], 'place_id'].tolist())


## 5. Fitur fasilitas C4 + activity tags
6 kelompok fasilitas via pencocokan kata utuh + 16 koreksi konteks agar false-positive (cerita sejarah, rencana, tetangga, keluhan) dibuang. `c4 = rata-rata 6 flag`; 0 berarti tidak ada bukti di deskripsi, bukan tidak ada fasilitas.


In [ ]:
FACILITY_KEYWORDS = {'toilet': ['toilet', 'kamar mandi', 'wc umum'],
    'parking': ['parkir', 'parkiran', 'parking'],
    'food': ['warung', 'restoran', 'rumah makan', 'kafe', 'cafe', 'kuliner'],
    'worship': ['mushola', 'musala', 'masjid', 'tempat ibadah', 'gereja', 'pura', 'vihara'],
    'accessibility': ['disabilitas', 'difabel', 'kursi roda', 'wheelchair', 'aksesibel'],
    'information_center': ['pusat informasi', 'information center', 'layanan informasi']}
ACTIVITY_KEYWORDS = {'hiking': ['hiking', 'mendaki', 'pendakian', 'trekking'],
    'fotografi': ['fotografi', 'spot foto', 'berfoto', 'pemandangan', 'panorama'],
    'snorkeling': ['snorkeling', 'snorkel'], 'diving': ['diving', 'menyelam', 'selam'],
    'camping': ['camping', 'berkemah', 'bumi perkemahan'],
    'kuliner': ['kuliner', 'makanan khas', 'jajanan', 'warung', 'restoran'],
    'sejarah': ['sejarah', 'bersejarah', 'peninggalan', 'museum', 'monumen'],
    'budaya': ['budaya', 'tradisi', 'kesenian', 'keraton'],
    'belanja': ['belanja', 'pusat perbelanjaan', 'pasar', 'mal', 'mall'],
    'berenang': ['berenang', 'kolam renang', 'waterpark', 'water park'],
    'edukasi': ['edukasi', 'pendidikan', 'belajar', 'museum'],
    'religi': ['ziarah', 'religi', 'ibadah', 'masjid', 'gereja', 'pura', 'vihara'],
    'rekreasi_keluarga': ['keluarga', 'wahana', 'taman bermain', 'taman hiburan']}
CATEGORY_TAGS = {'budaya': {'budaya'}, 'belanja': {'belanja'}, 'religi': {'religi'}, 'hiburan': {'rekreasi_keluarga'}}
desc = df['description'].map(normalize_text)
fac_cols = []
for fac, keys in FACILITY_KEYWORDS.items():
    c = 'facility_' + fac + '_mentioned'
    df[c] = desc.map(lambda t, ks=keys: int(any(contains_keyword(t, k) for k in ks)))
    fac_cols.append(c)
df['facility_keyword_score'] = df[fac_cols].mean(axis=1)
df['facility_description_missing'] = desc.eq('').astype(int)
CTX = {18: ('Museum Bank Indonesia', 'gereja ini dibongkar', ['worship'], 'Gereja dalam cerita sejarah telah dibongkar.'),
    57: ('Taman Lapangan Banteng', 'tidak jauh dari gereja katedral', ['worship'], 'Gereja adalah lokasi lain di dekat taman.'),
    94: ('Sumur Gumuling', 'pada masa lalu', ['worship'], 'Fungsi ibadah masa lalu, bukan fasilitas aktif terkonfirmasi.'),
    115: ('Monumen Sanapati', 'di taman depan gereja', ['worship'], 'Gereja adalah penanda lokasi di luar monumen.'),
    149: ('Goa Cerme', 'rencana pendirian masjid', ['worship'], 'Masjid disebut dalam cerita sejarah.'),
    176: ('Museum Gunung Merapi', 'ke depan juga akan dilengkapi', ['parking', 'information_center'], 'Parkir disebut sebagai rencana; layanan informasi adalah fungsi museum, bukan fasilitas terpisah yang dipastikan.'),
    213: ('Gedung Sate', 'gaya atap pura bali', ['worship'], 'Pura adalah perbandingan gaya arsitektur.'),
    232: ('Bukit Moko', 'kawasan bandung mampu menarik', ['food'], 'Kuliner dibahas sebagai daya tarik Bandung secara umum.'),
    249: ('Upside Down World Bandung', 'ruangan dengan tema', ['toilet'], 'Kamar mandi adalah ruangan bertema untuk foto.'),
    254: ('Teras Cikapundung BBWS', 'sebelumnya, taman ini', ['food'], 'Warung ada dalam cerita kondisi sebelumnya, bukan fasilitas saat ini yang dipastikan.'),
    270: ('Bukit Bintang', 'kuala lumpur, malaysia', ['food'], 'Deskripsi membahas tempat di negara lain; tidak dipakai sebagai bukti fasilitas record Bandung.'),
    295: ('Museum Nike Ardilla', 'hard rock cafe', ['food'], 'Cafe disebut sebagai perbandingan konsep desain.'),
    298: ('Gunung Lalakon', 'batu warung', ['food'], 'Batu Warung adalah nama batu, bukan warung makan.'),
    347: ('Taman Pandanaran', 'untuk mendapatkan parkir di sini cukup susah', ['parking'], 'Keluhan mencari parkir tidak mengonfirmasi area parkir destinasi.'),
    353: ('Taman Srigunting', 'di sisi barat terdapat gereja blenduk', ['worship'], 'Gereja adalah bangunan tetangga, bukan fasilitas taman.'),
    441: ('Pantai Jatimalang', 'bukan pintu masuk atau area parkir terverifikasi', ['parking'], 'Penyangkalan verifikasi koordinat, bukan bukti fasilitas parkir.')}
df['facility_review_note'] = ''
for pid, (nm, phr, rej, note) in CTX.items():
    hit = (df['place_id'].eq(pid) & df['place_name'].map(normalize_text).eq(normalize_text(nm)) & desc.str.contains(normalize_text(phr), regex=False))
    df.loc[hit, ['facility_' + k + '_mentioned' for k in rej]] = 0
    df.loc[hit, 'facility_review_note'] = note
df['c4_facility_score'] = df[fac_cols].mean(axis=1)
def tags(row):
    t = normalize_text(str(row['place_name']) + ' ' + str(row['description']))
    s = set(CATEGORY_TAGS.get(row['category_clean'], set()))
    for tag, keys in ACTIVITY_KEYWORDS.items():
        if any(contains_keyword(t, k) for k in keys):
            s.add(tag)
    return '|'.join(sorted(s))
df['activity_tags'] = df.apply(tags, axis=1)
df['feature_source_note'] = 'C4: indikator deskripsi dengan koreksi konteks tertentu; 0 bukan bukti fasilitas tidak tersedia. Activity_tags tetap heuristik.'
print('berbukti fasilitas:', int(df['c4_facility_score'].gt(0).sum()), '| koreksi konteks:', int(df['facility_review_note'].ne('').sum()))


## 6. Fitur K-Means + simpan
Z-score harga (cap P99) + rating model, one-hot kategori. Urutan kolom disamakan dengan pipeline kanonis, lalu simpan 2 CSV.


In [ ]:
num = df[['c1_ticket_price', 'c2_rating_for_model']].to_numpy(dtype=float, copy=True)
assert np.isfinite(num).all() and (num[:, 0] >= 0).all() and ((num[:, 1] >= 1) & (num[:, 1] <= 5)).all()
num[:, 0] = np.minimum(num[:, 0], cap_p99)
mean, scale = num.mean(axis=0), num.std(axis=0)
scale[scale == 0] = 1
cats = sorted(df['category_clean'].unique().tolist())
feat_cols = ['c1_ticket_price_z', 'c2_rating_z'] + ['category_' + c for c in cats]
onehot = np.array([[int(v == c) for c in cats] for v in df['category_clean']])
features = pd.DataFrame(np.column_stack(((num - mean) / scale, onehot)), columns=feat_cols)
features.insert(0, 'place_id', df['place_id'].to_numpy())
df['model_features_complete'] = True
COLUMN_ORDER = ['place_id', 'place_name', 'description', 'category', 'city', 'price', 'rating',
    'time_minutes', 'coordinate', 'lat', 'long', 'source_dataset', 'source_url', 'rating_status',
    'rating_source_url', 'rating_access_date', 'coordinate_review_required', 'coordinate_review_note',
    'data_quality_issues', 'province', 'category_original', 'category_clean', 'user_rating_mean',
    'user_rating_count', 'user_rating_std', 'c1_ticket_price', 'c2_rating', 'c2_rating_for_model',
    'rating_imputed', 'rating_model_note', 'facility_toilet_mentioned', 'facility_parking_mentioned',
    'facility_food_mentioned', 'facility_worship_mentioned', 'facility_accessibility_mentioned',
    'facility_information_center_mentioned', 'facility_keyword_score', 'facility_description_missing',
    'facility_review_note', 'c4_facility_score', 'activity_tags', 'feature_source_note', 'model_features_complete']
df = df[COLUMN_ORDER]
check_ids(df, 'final'); check_ids(features, 'fitur')
assert np.isfinite(features[feat_cols].to_numpy(dtype=float)).all()
(PROJECT_ROOT / OUT_DEST).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROJECT_ROOT / OUT_DEST, index=False, encoding='utf-8-sig', lineterminator='\n')
features.to_csv(PROJECT_ROOT / OUT_FEAT, index=False, encoding='utf-8-sig', lineterminator='\n')
print('tersimpan:', OUT_DEST, '|', OUT_FEAT)
print('ID lengkap 1-443 | rating diimputasi:', df.loc[df['rating_imputed'], 'place_id'].tolist())
display(features.head())
